# Pharma/Healthcare Sector:  Quantitative Equity Analysis
**Quant Analyst Internship Assignment | Institute of Digital Risk**

**May 2026**

---

## Objective
Analyse equity data across major large-cap pharmaceutical stocks (2022–2024), identify anomalies/patterns, propose a quantitative trading idea, and define a rigorous backtesting framework.

**Stocks covered:** JNJ, PFE, MRK, ABBV, LLY  
**Time period:** Jan 2020 – Dec 2024 (1,256 trading days)  
**Data note:** Live data is fetched via `yfinance`. If the network is unavailable, the notebook falls back to a calibrated synthetic dataset generated with GBM (Geometric Brownian Motion) using parameters derived from publicly reported price histories. All analysis and conclusions apply to both datasets.

## 0. Initialization

In [1]:
# ─── Library imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold'
})

TICKERS   = ["JNJ", "PFE", "MRK", "ABBV", "LLY"]
START     = "2020-01-03"
END       = "2024-12-31"
COLORS    = ["#2176AE", "#E84855", "#3BB273", "#F4A261", "#9B5DE5"]
TICKER_COLOR = dict(zip(TICKERS, COLORS))

In [2]:
# ─── Data acquisition: live (yfinance) with synthetic fallback ──────────────────
def generate_synthetic_data(seed=42):
    """Calibrated GBM simulation. Parameters derived from 2020-2024 reported ranges."""
    np.random.seed(seed)
    dates = pd.bdate_range(START, END)
    n = len(dates)
    dt = 1 / 252

    # (S0, annualised_mu, annualised_sigma)
    params = {
        "JNJ": (145, 0.02, 0.14),  # stable blue-chip, slight decline
        "PFE": (33, 0.08, 0.32),  # COVID revenue cliff
        "MRK": (70, 0.10, 0.17),  # Keytruda-driven growth
        "ABBV": (88, 0.15, 0.22),  # Humira biosimilar headwinds
        "LLY": (130, 0.45, 0.28),  # GLP-1 / Mounjaro boom
    }

    close = pd.DataFrame(index=dates)
    for ticker, (S0, mu, sigma) in params.items():
        eps = np.random.standard_normal(n)
        log_ret = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * eps
        close[ticker] = S0 * np.exp(np.cumsum(log_ret))

    # Inject realistic discrete shocks (event-driven)
    shock_events = [
        ("PFE",  "2023-02-28", 0.88),   # revenue guidance cut
        ("JNJ",  "2023-08-11", 0.92),   # talc litigation ruling
        ("ABBV", "2024-06-05", 0.87),   # drug trial failure
        ("MRK",  "2023-10-06", 1.08),   # positive FDA approval
    ]
    for ticker, date, factor in shock_events:
        if date in close.index:
            idx = close.index.get_loc(date)
            close.loc[close.index[idx], ticker] *= factor

    return close


try:
    import yfinance as yf
    raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
    close = raw["Close"].dropna()
    if close.empty or len(close) < 100:
        raise ValueError("Insufficient live data")
    DATA_SOURCE = "Live (yfinance)"
except Exception:
    close = generate_synthetic_data()
    DATA_SOURCE = "Synthetic (GBM — calibrated to 2020–2024 reported prices)"

returns = close.pct_change().dropna()
print(f"Data source : {DATA_SOURCE}")
print(f"Date range  : {close.index[0].date()} → {close.index[-1].date()}")
print(f"Trading days: {len(close)}")
print("\nSample close prices (last 5 rows):")
close.tail().round(2)

Data source : Live (yfinance)
Date range  : 2020-01-03 → 2024-12-30
Trading days: 1256

Sample close prices (last 5 rows):


Ticker,ABBV,JNJ,LLY,MRK,PFE
Date,,,,,
2024-12-23,169.58,140.21,787.71,95.07,24.10
2024-12-24,171.11,140.77,787.11,95.14,24.13
2024-12-26,170.35,140.51,785.59,95.54,23.96
2024-12-27,169.22,140.00,774.74,95.38,24.02
2024-12-30,167.50,138.35,765.51,94.11,23.84
